In [0]:
%sql
-- File: create_silver_routes_scd
-- Silver layer: cleaned GTFS routes with SCD Type 2 historization.
-- Source: dbr_dev.live_transit_monitor.bronze_gtfs_routes

CREATE TABLE IF NOT EXISTS dbr_dev.live_transit_monitor.silver_routes_scd (
    -- business key
    route_id            STRING    NOT NULL COMMENT 'GTFS route id; business key, joins to routeId in the GPS stream',

    -- identity attributes
    route_short_name    STRING             COMMENT 'Line number as shown to passengers',
    route_long_name     STRING             COMMENT 'Full route name, if published',
    route_desc          STRING             COMMENT 'Route description, if published',
    agency_id           STRING             COMMENT 'Operating agency',

    -- classification
    route_type          INT                COMMENT 'GTFS route type: 0/900 tram, 3/700 bus',
    route_type_name     STRING             COMMENT 'Human-readable route type, derived in silver',

    -- presentation attributes
    route_color         STRING             COMMENT 'Line colour, hex without #',
    route_text_color    STRING             COMMENT 'Text colour on the line colour, hex without #',

    -- change detection
    attributes_hash     STRING    NOT NULL COMMENT 'SHA-256 over tracked attributes, used to detect changes',

    -- SCD Type 2 historization
    valid_from          TIMESTAMP NOT NULL COMMENT 'Start of validity for this version',
    valid_to            TIMESTAMP          COMMENT 'End of validity; NULL means currently valid',
    is_current          BOOLEAN   NOT NULL COMMENT 'TRUE for the active version of the route',

    -- lineage
    source_file         STRING             COMMENT 'Source file the record came from',
    silver_ingestion_ts TIMESTAMP NOT NULL COMMENT 'When this version was written to silver'
)
USING DELTA
COMMENT 'GTFS routes with SCD Type 2 history. One row per version of a route.';

In [0]:
%sql
-- File: create_silver_routes_quarantine_scd
-- Rejected routes records, captured instead of silently dropped.

CREATE TABLE IF NOT EXISTS dbr_dev.live_transit_monitor.silver_routes_quarantine_scd (
    route_id            STRING             COMMENT 'Business key, may be NULL if that is why the row failed',
    raw_record          STRING             COMMENT 'Full source record as JSON, for investigation',
    rejection_reasons   ARRAY<STRING>      COMMENT 'Which quality rules the record violated',
    source_file         STRING             COMMENT 'Source file the record came from',
    quarantined_at      TIMESTAMP NOT NULL COMMENT 'When the record was rejected'
)
USING DELTA
COMMENT 'Quarantine for route records failing silver data-quality rules.';